In [1]:
import pandas as pd
import json

In [2]:
# Define file paths
passage_collection_path = "../data/raw/collection.tsv"
query_relevance_dev_path = "../data/raw/qrels_dev.tsv"
query_relevance_train_path = "../data/raw/qrels_train.tsv"
query_dev_path = "../data/raw/queries_dev.tsv"
query_eval_path = "../data/raw/queries_eval.tsv"
query_train_path = "../data/raw/queries_train.tsv"

# Read in the dataframes
passage_collection_df = pd.read_csv(passage_collection_path, sep="\t", names=["passage_id", "passage_text"])
query_relevance_dev_df = pd.read_csv(query_relevance_dev_path, sep="\t", names=["query_id", "query_relevance", "passage_id", "passage_relevance"])
query_relevance_train_df = pd.read_csv(query_relevance_train_path, sep="\t", names=["query_id", "query_relevance", "passage_id", "passage_relevance"])
query_dev_df = pd.read_csv(query_dev_path, sep="\t", names=["query_id", "query_text"])
query_eval_df = pd.read_csv(query_eval_path, sep="\t", names=["query_id", "query_text"])
query_train_df = pd.read_csv(query_train_path, sep="\t", names=["query_id", "query_text"])

In [3]:
# EDA
print("Passage Collection DataFrame Info:")
print(passage_collection_df.info())
print(passage_collection_df.head())

Passage Collection DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8841823 entries, 0 to 8841822
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   passage_id    int64 
 1   passage_text  object
dtypes: int64(1), object(1)
memory usage: 134.9+ MB
None
   passage_id                                       passage_text
0           0  The presence of communication amid scientific ...
1           1  The Manhattan Project and its atomic bomb help...
2           2  Essay on The Manhattan Project - The Manhattan...
3           3  The Manhattan Project was the name for a proje...
4           4  versions of each volume as well as complementa...


In [5]:
print("Query Relevance Dev DataFrame Info:")
print(query_relevance_dev_df.info())
print(query_relevance_dev_df.head())

print("\nQuery Relevance Train DataFrame Info:")
print(query_relevance_train_df.info())
print(query_relevance_train_df.head())

Query Relevance Dev DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59273 entries, 0 to 59272
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   query_id           59273 non-null  int64
 1   query_relevance    59273 non-null  int64
 2   passage_id         59273 non-null  int64
 3   passage_relevance  59273 non-null  int64
dtypes: int64(4)
memory usage: 1.8 MB
None
   query_id  query_relevance  passage_id  passage_relevance
0   1102432                0     2026790                  1
1   1102431                0     7066866                  1
2   1102431                0     7066867                  1
3   1090282                0     7066900                  1
4     39449                0     7066905                  1

Query Relevance Train DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 532761 entries, 0 to 532760
Data columns (total 4 columns):
 #   Column             Non-

In [6]:
print("Query Dev DataFrame Info:")
print(query_dev_df.info())
print(query_dev_df.head())

print("\nQuery Eval DataFrame Info:")
print(query_eval_df.info())
print(query_eval_df.head())

print("\nQuery Train DataFrame Info:")
print(query_train_df.info())
print(query_train_df.head())


Query Dev DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101093 entries, 0 to 101092
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   query_id    101093 non-null  int64 
 1   query_text  101093 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.5+ MB
None
   query_id                      query_text
0   1048578  cost of endless pools/swim spa
1   1048579                    what is pcnt
2   1048580               what is pcb waste
3   1048581                   what is pbis?
4   1048582                  what is paysky

Query Eval DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101092 entries, 0 to 101091
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   query_id    101092 non-null  int64 
 1   query_text  101092 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.5+ MB
None
   query_id             

In [7]:

# Check the if the `query_id` in the relevance dataframes are unique (i.e. if the len of unique values is the same as the len of the dataframe)
# If they are not unique, means that there are multiple passages relevant to the same query
qrel_dev_unique_queries = query_relevance_dev_df["query_id"].nunique()
qrel_dev_total_queries = len(query_relevance_dev_df)
print(f"\nQuery Relevance Dev DataFrame - Unique Queries: {qrel_dev_unique_queries}, Total Queries: {qrel_dev_total_queries}")

qrel_train_unique_queries = query_relevance_train_df["query_id"].nunique()
qrel_train_total_queries = len(query_relevance_train_df)
print(f"Query Relevance Train DataFrame - Unique Queries: {qrel_train_unique_queries}, Total Queries: {qrel_train_total_queries}")



Query Relevance Dev DataFrame - Unique Queries: 55578, Total Queries: 59273
Query Relevance Train DataFrame - Unique Queries: 502939, Total Queries: 532761


In [8]:
# Check for data leakage between train and dev sets
train_query_ids = set(query_train_df["query_id"].unique())
dev_query_ids = set(query_dev_df["query_id"].unique())
leakage_query_ids = train_query_ids.intersection(dev_query_ids)
print(f"\nNumber of overlapping query IDs between train and dev sets: {len(leakage_query_ids)}")


Number of overlapping query IDs between train and dev sets: 0


In [9]:
# Check for queries in dev set that are not in relevance dev set
dev_query_ids = set(query_dev_df["query_id"].unique())
relevance_dev_query_ids = set(query_relevance_dev_df["query_id"].unique())
missing_in_relevance_dev = dev_query_ids - relevance_dev_query_ids
print(f"Number of query IDs in dev set not in relevance dev set: {len(missing_in_relevance_dev)}")

# Check for queries in train set that are not in relevance train set
train_query_ids = set(query_train_df["query_id"].unique())
relevance_train_query_ids = set(query_relevance_train_df["query_id"].unique())
missing_in_relevance_train = train_query_ids - relevance_train_query_ids
print(f"Number of query IDs in train set not in relevance train set: {len(missing_in_relevance_train)}")

Number of query IDs in dev set not in relevance dev set: 45515
Number of query IDs in train set not in relevance train set: 305792


In [13]:
# Check for passages that are in relevance dataframes but not in passage collection
passage_ids_in_collection = set(passage_collection_df["passage_id"].unique())
passage_ids_in_relevance_dev = set(query_relevance_dev_df["passage_id"].unique())
missing_passages_in_collection_dev = passage_ids_in_relevance_dev - passage_ids_in_collection
print(f"Number of passage IDs in relevance dev set not in passage collection: {len(missing_passages_in_collection_dev)}")

passage_ids_in_relevance_train = set(query_relevance_train_df["passage_id"].unique())
missing_passages_in_collection_train = passage_ids_in_relevance_train - passage_ids_in_collection
print(f"Number of passage IDs in relevance train set not in passage collection: {len(missing_passages_in_collection_train)}")

Number of passage IDs in relevance dev set not in passage collection: 0
Number of passage IDs in relevance train set not in passage collection: 0


In [10]:
# Filter for queries in dev set that are in relevance dev set
filtered_query_dev_df = query_dev_df[query_dev_df["query_id"].isin(relevance_dev_query_ids)]
print(f"Filtered Query Dev DataFrame Length (only queries present in relevance dev set):")
print(len(filtered_query_dev_df))

# Filter for queries in train set that are in relevance train set
filtered_query_train_df = query_train_df[query_train_df["query_id"].isin(relevance_train_query_ids)]
print(f"\nFiltered Query Train DataFrame Length (only queries present in relevance train set):")
print(len(filtered_query_train_df))

Filtered Query Dev DataFrame Length (only queries present in relevance dev set):
55578

Filtered Query Train DataFrame Length (only queries present in relevance train set):
502939


In [11]:
# Create a mapping of query to relevant passages for fast lookup
query_relevance_dev_dict = (query_relevance_dev_df.groupby("query_id")["passage_id"]
                            .apply(lambda x: [str(pid) for pid in x])
                            .to_dict())
query_relevance_train_dict = (query_relevance_train_df.groupby("query_id")["passage_id"]
                              .apply(lambda x: [str(pid) for pid in x])
                              .to_dict())

# For each query in dev set, we want to format it as:
# {
#   "qid": "1234",
#   "query": "how to change iphone battery",
#   "qrels": ["201", "55689"]  // from qrels.dev
# }

formatted_dev_data_output_path = "../data/processed/formatted_dev_data.jsonl"

formatted_dev_data = []
for _, row in filtered_query_dev_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = query_relevance_dev_dict.get(qid, [])
    formatted_dev_data.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

# Save formatted dev data to be JSONL
with open(formatted_dev_data_output_path, "w") as f:
    for item in formatted_dev_data:
        f.write(json.dumps(item) + "\n")


In [12]:
formatted_train_data_output_path = "../data/processed/formatted_train_data.jsonl"

formatted_train_data = []
for _, row in filtered_query_train_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = query_relevance_train_dict.get(qid, [])
    formatted_train_data.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

# Save formatted train data to be JSONL
with open(formatted_train_data_output_path, "w") as f:
    for item in formatted_train_data:
        f.write(json.dumps(item) + "\n")